# RouterV3 -- environment bring-up + end-to-end verification

Builds a headless bridge to KiCad's real interactive router (`PNS::ROUTER`) by compiling a small pybind11 module *inside* a KiCad 9.0.8 source checkout, then proves it actually works by routing a real net on a toy board and independently checking the result.

**Why compile KiCad from source at all?** `PNS::ROUTER` has no Python API. The only way to drive it is to link directly against KiCad's internal C++ engine libraries (`pnsrouter`, `pcbcommon`, ...), which only exist as build artifacts inside a KiCad source build -- there's no prebuilt package that exposes them externally. See `docs/engine_access.md` for the full rationale (and why the public `kicad-python` IPC API doesn't work for this -- it doesn't expose routing at all).

Works on Google Colab or any Linux Jupyter kernel with `sudo`.

**Status: unverified past the OpenGL fix.** The C++ in `pcbworld/engine/cpp/` was written from reading KiCad's source, not compiled by me -- I don't have a Linux build environment to test it myself. Design choice to reduce iteration pain: **step 3 below is a single atomic script**, not a chain of small cells. Jupyter/Colab keeps running later cells after an earlier one errors, which silently produces confusing downstream failures (this bit us twice with small cells). One script with `set -e` and `=== STAGE ===` checkpoints means a failure stops immediately and the full log up to that point is right there in one place -- paste the whole cell output back to me, not just the traceback at the bottom, and I can see exactly which command failed and why the first time.

Time budget: ~10 min installing build deps, ~30-60 min compiling (first run only -- see the Drive-caching note in step 3).

## 1. Configuration

In [ ]:
import os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

WORKDIR = Path("/content") if IN_COLAB else Path.home() / "routerv3-build"
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

REPO_URL = "https://github.com/Klutzhehe/Routerv3.git"
KICAD_TAG = "9.0.8"

# Cache the built KiCad+bridge tree on Drive so future sessions can skip the
# ~30-60 min compile. Caches the whole kicad-src dir (source + build/
# together, not just build/) -- restoring build/ alone would break, because
# a fresh git clone resets file timestamps to "now", which is newer than
# the cached object files, so ninja would think everything's stale and
# rebuild anyway. Keeping source and build together preserves their
# relative timestamps across a restore.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_CACHE_DIR = Path("/content/drive/MyDrive/routerv3_cache")
else:
    DRIVE_CACHE_DIR = WORKDIR / "cache"
DRIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CACHE_TARBALL = DRIVE_CACHE_DIR / f"kicad-src-{KICAD_TAG}.tar.gz"

os.environ["WORKDIR"] = str(WORKDIR)
os.environ["REPO_URL"] = REPO_URL
os.environ["KICAD_TAG"] = KICAD_TAG
os.environ["DRIVE_CACHE_TARBALL"] = str(DRIVE_CACHE_TARBALL)
print(f"WORKDIR={WORKDIR}  IN_COLAB={IN_COLAB}")
print(f"DRIVE_CACHE_TARBALL={DRIVE_CACHE_TARBALL}")

## 2. Clone/update this repo
Always run this before step 3, even on a repeat session -- picks up anything I've since pushed instead of building against stale code.

In [ ]:
%%bash
set -e
cd "$WORKDIR"
if [ ! -d Routerv3 ]; then
  git clone "$REPO_URL" Routerv3
else
  cd Routerv3 && git pull
fi

## 3. Build everything (one atomic script)

Does, in order, stopping immediately on the first failure: install system build deps -> restore a cached build from Drive if one exists -> clone KiCad source (skipped if restored) -> wire our bridge into KiCad's CMake tree -> configure (skipped if restored) -> build just the `pcbworld_pns_bridge` target -> save the result back to Drive.

**Re-running after a previous attempt failed partway through:** safe to just re-run this cell -- every stage is idempotent (checks before doing work), *except* if a previous run left a broken/partial `build/` dir from a `cmake configure` that started but then errored (e.g. a missing system package caught mid-configure). In that case run `!rm -rf "$WORKDIR/kicad-src/build"` first so this cell doesn't mistake the broken partial config for a completed one.

**Drive caching:** automatic, no manual steps -- step 1 mounts Drive and this cell restores/saves `$DRIVE_CACHE_TARBALL` (the whole `kicad-src` dir, source + build together) on its own. First run compiles for real (~30-60 min) and caches it; later runs/sessions restore from Drive and skip straight to a quick incremental build (only rebuilds if the bridge source changed).

In [ ]:
%%bash
set -e

echo "=== STAGE: apt build dependencies ==="
sudo add-apt-repository -y ppa:kicad/kicad-9.0-releases

# `apt-get build-dep` needs deb-src entries, which are disabled by default
# both in Colab's base sources.list and in the PPA's own (auto-added,
# commented-out) deb-src line. Enable whatever's there commented-out...
sudo sed -i -E 's/^#\s*(deb-src .*)/\1/' /etc/apt/sources.list /etc/apt/sources.list.d/*.list || true
# ...and if the base image stripped deb-src out entirely rather than just
# commenting it, fall back to mirroring every deb line as deb-src.
if ! grep -rq '^deb-src ' /etc/apt/sources.list; then
  grep '^deb ' /etc/apt/sources.list | sed 's/^deb /deb-src /' | sudo tee -a /etc/apt/sources.list >/dev/null
fi

sudo apt-get update -qq
sudo apt-get build-dep -y kicad
sudo apt-get install -y kicad cmake ninja-build
# `apt-get build-dep` didn't pull these on a prior Colab run even though
# KiCad's CMakeLists.txt does a hard find_package(OpenGL REQUIRED) for the
# `gal` library we link against -- install explicitly rather than trusting
# build-dep to have covered it.
sudo apt-get install -y libgl1-mesa-dev libglu1-mesa-dev libglvnd-dev mesa-common-dev
pip install -q pybind11

echo "=== STAGE: restore cached KiCad build from Drive (if available) ==="
cd "$WORKDIR"
if [ ! -d kicad-src ] && [ -f "$DRIVE_CACHE_TARBALL" ]; then
  echo "found $DRIVE_CACHE_TARBALL, restoring (skips clone + most of configure/build)"
  tar xzf "$DRIVE_CACHE_TARBALL"
else
  echo "no cache to restore (first run, or already have a local kicad-src)"
fi

echo "=== STAGE: clone KiCad source (tag $KICAD_TAG) ==="
if [ ! -d kicad-src ]; then
  git clone --depth 1 --branch "$KICAD_TAG" https://gitlab.com/kicad/code/kicad.git kicad-src
fi

echo "=== STAGE: wire pcbworld bridge into KiCad's CMake tree ==="
rm -rf kicad-src/pcbworld_bridge
cp -r Routerv3/pcbworld/engine/cpp kicad-src/pcbworld_bridge
MARKER='add_subdirectory( ${CMAKE_SOURCE_DIR}/pcbworld_bridge'
if ! grep -qF "$MARKER" kicad-src/pcbnew/CMakeLists.txt; then
  printf '\nadd_subdirectory( ${CMAKE_SOURCE_DIR}/pcbworld_bridge ${CMAKE_BINARY_DIR}/pcbworld_bridge )\n' >> kicad-src/pcbnew/CMakeLists.txt
fi

echo "=== STAGE: cmake configure (Release) ==="
cd "$WORKDIR/kicad-src"
# pip-installed pybind11 doesn't sit on CMake's default search path;
# `python3 -m pybind11 --cmakedir` is pybind11's own documented way to get
# the directory containing pybind11Config.cmake for find_package(CONFIG).
PYBIND11_CMAKE_DIR="$(python3 -m pybind11 --cmakedir)"
echo "pybind11 cmake dir: $PYBIND11_CMAKE_DIR"

if [ ! -f build/CMakeCache.txt ]; then
  cmake -S . -B build -G Ninja \
    -DCMAKE_BUILD_TYPE=Release \
    -DKICAD_BUILD_QA_TESTS=OFF \
    -DKICAD_SCRIPTING_WXPYTHON=OFF \
    -DKICAD_BUILD_I18N=OFF \
    -DKICAD_USE_CMAKE_FINDPROTOBUF=ON \
    -Dpybind11_DIR="$PYBIND11_CMAKE_DIR"
else
  echo "build/CMakeCache.txt already exists, skipping configure (rm -rf build to force a clean reconfigure)"
fi

echo "=== STAGE: build pcbworld_pns_bridge ==="
cmake --build build --target pcbworld_pns_bridge -j"$(nproc)"

echo "=== STAGE: save build to Drive cache ==="
cd "$WORKDIR"
tar czf "$DRIVE_CACHE_TARBALL.tmp" kicad-src
mv "$DRIVE_CACHE_TARBALL.tmp" "$DRIVE_CACHE_TARBALL"
echo "cached to $DRIVE_CACHE_TARBALL ($(du -h "$DRIVE_CACHE_TARBALL" | cut -f1))"

echo "=== DONE: bridge built successfully ==="
find kicad-src/build -iname 'pcbworld_pns_bridge*.so'

## 4. Import the compiled bridge

In [ ]:
import sys, glob

matches = glob.glob(f"{WORKDIR}/kicad-src/build/**/pcbworld_pns_bridge*.so", recursive=True)
assert matches, "build didn't produce the extension -- check step 3's output for the STAGE it stopped at"
sys.path.insert(0, str(Path(matches[0]).parent))

import pcbworld_pns_bridge as bridge

print(bridge.__file__)
print("bridge module loaded OK")

## 5. End-to-end verification: route a real net on a toy board

This is the actual proof the whole stack works, not just "it imported":

1. Generate a 2-pad, 1-net board with the **system** `pcbnew` module
   (`scripts/make_toy_board.py`) -- independent of the bridge we're testing.
2. Load it with our bridge, find both pads via `query_hover_items`, route
   between them, commit, save.
3. Reload the saved file with the **system** `pcbnew` module again
   (`scripts/verify_routed_board.py`) and assert a track actually exists
   connecting the two pads' net. This catches the failure mode where our
   bridge *thinks* it committed a route but the board file on disk doesn't
   actually reflect it.

**Steps 1 and 3 both run as separate `python3` subprocesses (`%%bash`),
not inline Python in this kernel.** Confirmed the hard way: importing the
system `pcbnew` module in the *same* process that already has
`pcbworld_pns_bridge` loaded crashes -- both statically link large,
overlapping chunks of KiCad's own C++ code and both define KiCad's
"exactly one instance per process" globals (`Kiface()`, `GFootprintTable`).
This isn't just a notebook quirk -- it applies to the RL environment too,
see `docs/performance.md` and `docs/engine_access.md`.

In [ ]:
%%bash
set -e
cd "$WORKDIR"
python3 Routerv3/scripts/make_toy_board.py toy_board.kicad_pcb

In [ ]:
board_path = str(WORKDIR / "toy_board.kicad_pcb")

b = bridge.PNSBridge()
assert b.load_board(board_path), "load_board failed"

nets = b.net_names()
print("nets:", nets)
assert "toy_net" in nets

# Pad positions match scripts/make_toy_board.py: (0, 0) mm and (20, 0) mm.
MM = 1_000_000  # KiCad internal units are nm
pad_a = b.query_hover_items(0, 0, layer=0, slop_radius=int(0.5 * MM))
pad_b = b.query_hover_items(20 * MM, 0, layer=0, slop_radius=int(0.5 * MM))
print("pad_a candidates:", [(c.kind, c.x, c.y) for c in pad_a])
print("pad_b candidates:", [(c.kind, c.x, c.y) for c in pad_b])
assert pad_a and pad_b, "couldn't find both pads via query_hover_items"

b.set_mode(bridge.MODE_ROUTE_SINGLE)
b.set_track_width(int(0.25 * MM))

assert b.start_route(pad_a[0].x, pad_a[0].y, pad_a[0].id, 0), "start_route failed"
assert b.push(10 * MM, 0, -1), "push failed"
assert b.fix(pad_b[0].x, pad_b[0].y, pad_b[0].id, False, False), "fix failed"
b.commit_routing()
assert b.save_board(board_path), "save_board failed"
print("routed and saved OK")

## 6. Headless DRC

Runs KiCad's real `DRC_ENGINE` (same one the GUI's DRC dialog drives) against the routed toy board -- no dialog, no rules file (falls back to KiCad's built-in defaults). New/untested code path: first run may surface link errors (most likely from `pcbnew_kiface_objects`, newly added in `pcbworld/engine/cpp/CMakeLists.txt` for this) -- if step 3 fails here, paste the linker output back rather than guessing further ahead of it.

In [ ]:
violations = b.run_drc()
print(f"{len(violations)} DRC violation(s)")
for v in violations:
    print(f"  [{v.severity}] code={v.error_code} ({v.x}, {v.y}): {v.message}")

# The toy board is two pads joined by one clean track with plenty of
# clearance -- errors here would mean either a real DRC finding (routing
# bug) or a DRC_ENGINE misconfiguration (e.g. missing default rules
# because we never loaded a PROJECT). Print rather than hard-assert
# zero-violations on the first run so a false positive is visible instead
# of just failing the cell.
errors = [v for v in violations if v.severity == "error"]
if errors:
    print("NOTE: unexpected error-severity violations on a clean toy board -- inspect above")


In [ ]:
%%bash
set -e
cd "$WORKDIR"
python3 Routerv3/scripts/verify_routed_board.py toy_board.kicad_pcb toy_net